# Task 0 [3 points]
For all tasks in this assignment, use the RecBole package (https://recbole.io/) and the Amazon
Beauty datasets. Get familiar with the package. For using the package for the entire pipeline (loading
and splitting the data, training the model, evaluation), you can obtain 3 points. These points may
be added to any of the following tasks.

# Task 1 [2 points]
1. Train and evaluate the SASRec model. Remember to split the data properly for the sequential
recommendations (last user interaction for testing, second last user interaction for validation).
Calculate the metrics (at least recall and nDCG) for k = 5, 10, 20 (note that you don’t need
to calculate the recommendations multiple times to do so).
2. Analyze the quality of recommendations for users with different sequence lengths:
- After performing the standard evaluation, divide the data based on the input sequence
length (e.g. 0 − 5, 5 − 10, 15 − 20, 20 − 30, 30+).
- Calculate how many sequences fall into each group.
- Calculate the metrics for each group.
- Visualize the results in a clear form. Describe your observations.

In [2]:
import matplotlib.pyplot as plt
import numpy as np

import pandas as pd
import torch

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_seed

# np.float_ = np.float64
# np.complex_ = np.complex128
# np.unicode_ = np.str_

In [3]:
default_config_dict = {
    # dataset config : Sequential Recommendation
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {"inter": ["user_id", "item_id", "timestamp"]},  # load user-item interactions with timestamp
    "ITEM_LIST_LENGTH_FIELD": "item_length",
    "LIST_SUFFIX": "_list",
    "MAX_ITEM_LIST_LENGTH": 35,  # maximum length of the item sequence for each user

    # model config
    "embedding_size": 64,
    "hidden_size": 128,
    "num_layers": 1,
    "dropout_prob": 0.3,
    "loss_type": 'CE',

    # Training and evaluation config
    "epochs": 500,
    "train_batch_size": 4096,
    "eval_batch_size": 4096,
    "train_neg_sample_args": None,
    "eval_args": {
        "group_by": "user",
        "order": "TO",
        "split": {"LS": "valid_and_test"},
        "mode": "full"
    },
    "metrics": ["Recall", "MRR", "NDCG", "Hit", "Precision"],
    "topk": [5, 10, 20],
    "valid_metric": "MRR@10"
}


In [ ]:
config = Config(model="SASRec", dataset="amazon-beauty", config_dict=default_config_dict)
init_seed(config["seed"], config["reproducibility"])

dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, show_progress=True)
test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=True)

print("Best validation score:", best_valid_score)
print("Best validation result:", best_valid_result)
print("Test result:", test_result)


## Task 1.2: recommendation quality by input sequence length

In [ ]:
def evaluate_by_sequence_length(trainer, test_data, topk=(5, 10, 20)):
    """Evaluate full-sort sequential recommendations grouped by input item_length."""
    max_k = max(topk)
    rows = []

    trainer.model.eval()
    trainer.tot_item_num = test_data._dataset.item_num
    if getattr(trainer, "item_tensor", None) is None:
        trainer.item_tensor = test_data._dataset.get_item_feature().to(trainer.device)

    discounts = 1.0 / torch.log2(torch.arange(max_k, device=trainer.device, dtype=torch.float32) + 2.0)

    with torch.no_grad():
        for batched_data in test_data:
            interaction, scores, positive_u, positive_i = trainer._full_sort_batch_eval(batched_data)
            positive_u = torch.as_tensor(positive_u, device=trainer.device)
            positive_i = torch.as_tensor(positive_i, device=trainer.device)

            top_items = torch.topk(scores, k=max_k, dim=1).indices
            hits = top_items[positive_u].eq(positive_i.unsqueeze(1))
            item_lengths = interaction[trainer.config["ITEM_LIST_LENGTH_FIELD"]].to(trainer.device)[positive_u]

            batch = {"item_length": item_lengths.detach().cpu().numpy()}
            for k in topk:
                hit_at_k = hits[:, :k].any(dim=1).float()
                ndcg_at_k = (hits[:, :k].float() * discounts[:k]).max(dim=1).values
                batch[f"recall@{k}"] = hit_at_k.detach().cpu().numpy()
                batch[f"ndcg@{k}"] = ndcg_at_k.detach().cpu().numpy()

            rows.append(pd.DataFrame(batch))

    per_sequence = pd.concat(rows, ignore_index=True)

    bins = [0, 5, 10, 15, 20, 30, np.inf]
    labels = ["0-5", "6-10", "11-15", "16-20", "21-30", "31+"]
    per_sequence["length_group"] = pd.cut(
        per_sequence["item_length"], bins=bins, labels=labels, include_lowest=True, right=True
    )

    aggregations = {"item_length": "size"}
    for k in topk:
        aggregations[f"recall@{k}"] = "mean"
        aggregations[f"ndcg@{k}"] = "mean"

    by_length = (
        per_sequence.groupby("length_group", observed=False)
        .agg(aggregations)
        .rename(columns={"item_length": "n_sequences"})
        .reset_index()
    )
    return per_sequence, by_length


per_sequence_metrics, length_group_metrics = evaluate_by_sequence_length(trainer, test_data)
length_group_metrics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(length_group_metrics["length_group"].astype(str), length_group_metrics["n_sequences"])
axes[0].set_title("Number of test sequences by input length")
axes[0].set_xlabel("Input sequence length")
axes[0].set_ylabel("Number of sequences")

for metric, linestyle in [("recall", "-"), ("ndcg", "--")]:
    for k in [5, 10, 20]:
        axes[1].plot(
            length_group_metrics["length_group"].astype(str),
            length_group_metrics[f"{metric}@{k}"],
            marker="o",
            linestyle=linestyle,
            label=f"{metric}@{k}",
        )

axes[1].set_title("SASRec quality by input sequence length")
axes[1].set_xlabel("Input sequence length")
axes[1].set_ylabel("Metric value")
axes[1].legend(ncol=2)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:
best_recall_group = length_group_metrics.loc[length_group_metrics["recall@20"].idxmax()]
best_ndcg_group = length_group_metrics.loc[length_group_metrics["ndcg@20"].idxmax()]

print(
    f"Most test sequences are in the "
    f"{length_group_metrics.loc[length_group_metrics['n_sequences'].idxmax(), 'length_group']} length group."
)
print(
    f"The best Recall@20 is for length group {best_recall_group['length_group']} "
    f"({best_recall_group['recall@20']:.4f})."
)
print(
    f"The best NDCG@20 is for length group {best_ndcg_group['length_group']} "
    f"({best_ndcg_group['ndcg@20']:.4f})."
)
print(
    "Compare the curves above: if longer histories have higher metrics, SASRec benefits from more context; "
    "if the largest buckets flatten or drop, the model may be limited by sparse examples in those groups."
)


# Task 2 [2 points]
1. Train and evaluate the LightGCN model. Use a proper data split. Calculate the metrics (at least
recall and nDCG) for k = 5, 10, 20 (note that you don’t need to calculate the recommendations
multiple times to do so).
2. Analyze the popularity of items:
- Calculate the popularity of each item. Make a bar plot with ordered popularities (from
largest to smallest).
- Analyze the popularity of the items you have recommended: how popular are the items
you recommend, and how often items of different popularity are recommended.
- By doing this analysis, you can observe the Matthew effect for Recommender Systems.
Explain it.